# TraceCir — P0 (TAPR baseline E0) trên Vast.ai

Bản chạy trên máy thuê Vast.ai (Linux, 1 GPU, Jupyter). Khác bản Colab:
- **Không dùng Google Drive.** Mọi thứ nằm trên ổ đĩa của máy thuê (`/workspace`).
- Ổ đĩa Vast **không nới được sau khi thuê**, nên cell tải COCO xóa file `.zip` ngay sau khi giải nén.
- Cache đặc trưng vẫn **resume được** nếu bị ngắt (chạy lại đúng cell).
- Compiler Qwen chạy **bf16 không nén** (cần GPU ≥ 20GB VRAM, ví dụ RTX 3090/4090). Nếu GPU chỉ 12GB, đổi `LOAD_IN_4BIT = True` ở Bước 4.

> ⚠️ **Tiền tính theo giờ máy còn tồn tại, kể cả lúc rảnh.** Chạy xong: tải file trong `/workspace/outputs/` về máy, rồi **Destroy** máy trên trang Vast.


## Bước 0 — Kiểm tra máy

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /workspace | tail -1
!free -h | head -2
!nproc

## Bước 1 — Lấy code + cài thư viện

In [ ]:
import os

REPO_URL = 'https://github.com/khaidz123321/TraceCir.git'
REPO_DIR = '/workspace/TraceCir'

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd /workspace/TraceCir
%pip install -q -r requirements.txt

## Bước 2 — Dữ liệu CIRCO

Tải COCO `unlabeled2017.zip` (~19.5GB), giải nén, **xóa zip ngay** để tiết kiệm ổ đĩa (đỉnh điểm ~39GB nếu giữ cả hai).

In [ ]:
import os

DATA_DIR = '/workspace/data/CIRCO'
IMG_DIR = f'{DATA_DIR}/COCO2017_unlabeled/unlabeled2017'
ZIP_PATH = '/workspace/unlabeled2017.zip'
N_IMAGES = 123403

if os.path.isdir(IMG_DIR) and len(os.listdir(IMG_DIR)) == N_IMAGES:
    print(f'Anh da giai nen du {N_IMAGES} file, bo qua.')
else:
    if not os.path.exists(ZIP_PATH):
        print('Dang tai COCO (~19.5GB)...')
        !wget -q --show-progress -c http://images.cocodataset.org/zips/unlabeled2017.zip -O {ZIP_PATH}
    print('Dang giai nen...')
    os.makedirs(f'{DATA_DIR}/COCO2017_unlabeled', exist_ok=True)
    !unzip -q -o {ZIP_PATH} -d {DATA_DIR}/COCO2017_unlabeled/
    n = len(os.listdir(IMG_DIR))
    print(f'Da giai nen {n} anh (ky vong {N_IMAGES}).')
    assert n == N_IMAGES, 'Giai nen thieu anh - tai/giai nen lai'
    !rm -f {ZIP_PATH}
    print('Da xoa file zip de tiet kiem o dia.')
!df -h /workspace | tail -1

In [ ]:
import os

if not os.path.isdir(f'{DATA_DIR}/annotations'):
    if not os.path.isdir('/workspace/CIRCO_annotations_repo'):
        !git clone -q https://github.com/miccunifi/CIRCO.git /workspace/CIRCO_annotations_repo
    !cp -r /workspace/CIRCO_annotations_repo/annotations {DATA_DIR}/annotations
print(os.listdir(f'{DATA_DIR}/annotations'))

## Bước 3 — Cache đặc trưng OpenCLIP

Trên RTX 3090 ước tính ~35–40 phút (chưa đo). Nếu bị ngắt, **chạy lại đúng cell này** sẽ tiếp tục từ chỗ dở. "Đã xong" được xác định bằng `_progress.json`, không dựa vào kích thước file `.npy` (file được cấp sẵn full dung lượng nên luôn trông như đã đầy).

In [ ]:
import sys, os, json, numpy as np
sys.path.insert(0, '/workspace/TraceCir')

from src.models.openclip_utils import load_openclip
from src.data.datasets import CIRCODataset
from src.p0.feature_cache import build_feature_cache, FeatureCache

device = 'cuda'
model, preprocess, tokenizer = load_openclip(device=device)

ds_classic = CIRCODataset(DATA_DIR, 'val', 'classic', preprocess)
TOTAL = len(ds_classic)
print('So anh trong index:', TOTAL)

FEATURE_CACHE_DIR = '/workspace/features/circo'
FILES = ['global.npy', 'local.npy', 'image_ids.json', '_progress.json']


def cache_completed(cache_dir):
    if not all(os.path.exists(f'{cache_dir}/{f}') for f in FILES):
        return 0
    return json.load(open(f'{cache_dir}/_progress.json'))['completed']


done = cache_completed(FEATURE_CACHE_DIR)
print(f'Tien do hien tai: {done}/{TOTAL}')
if done >= TOTAL:
    print('Cache da xong day du, bo qua tinh toan.')
else:
    build_feature_cache(
        model, ds_classic,
        output_dir=FEATURE_CACHE_DIR,
        id_key='image_id',
        device=device,
        batch_size=128,
        num_workers=6,
    )

In [ ]:
# Kiem tra cache xong THAT (doc _progress.json, khong dung len(cache))
done = cache_completed(FEATURE_CACHE_DIR)
assert done == TOTAL, f'Cache chua xong: {done}/{TOTAL}. Chay lai cell Buoc 3.'

cache = FeatureCache(FEATURE_CACHE_DIR)
print('global:', cache.global_features.shape, cache.global_features.dtype)
print('local :', cache.local_features.shape, cache.local_features.dtype)
idx = np.sort(np.random.default_rng(0).choice(TOTAL, 200, replace=False))
norms = np.linalg.norm(np.asarray(cache.global_features[idx]), axis=1)
assert norms.min() > 0.5, 'Co dong global trong - cache khong day du'
print('OK - cache day du.')

## Bước 4 — Transition Compiler (Qwen2.5-VL-7B-Instruct)

Chạy trên ~220 câu truy vấn CIRCO validation. **Sau khi chạy xong, bắt buộc tự audit thủ công** (mục 5.3 giao thức P0) trước khi chạy Bước 5.

In [ ]:
import torch
from src.p0.compiler import load_compiler, run_compiler_batch

LOAD_IN_4BIT = False   # True neu GPU chi co ~12GB VRAM

compiler_model, compiler_processor = load_compiler(
    model_name='Qwen/Qwen2.5-VL-7B-Instruct',
    device='cuda',
    load_in_4bit=LOAD_IN_4BIT,
)
print('VRAM dang dung (GB):', round(torch.cuda.memory_allocated() / 1e9, 1))

In [ ]:
from src.data.datasets import CIRCODataset

# 'relative' nhung KHONG qua preprocess: compiler can anh PIL goc
ds_query = CIRCODataset(DATA_DIR, 'val', 'relative', preprocess=lambda x: x)
print('So cau truy van CIRCO val:', len(ds_query))

queries = [
    (item['reference_image'], item['relative_caption'], str(item['reference_img_id']))
    for item in ds_query
]

os.makedirs('/workspace/outputs', exist_ok=True)
COMPILED_PATH = '/workspace/outputs/circo_compiled.jsonl'
records = run_compiler_batch(compiler_model, compiler_processor, queries, COMPILED_PATH)
print(f'Da chay compiler cho {len(records)} cau, luu tai {COMPILED_PATH}')
print('Khong parse duoc:', sum(1 for r in records if not r['parse_ok']))

### ⚠️ Audit thủ công (bắt buộc, không tự động hóa được)

Đọc ít nhất 100 dòng, tự gán nhãn Correct / Partially correct / Incorrect cho `target` và `atoms` so với `modification`. Nếu Correct < ~85%: sửa `COMPILER_PROMPT` trong `src/p0/compiler.py`, `git push`, `git pull` lại trong máy này, rồi chạy lại Bước 4.

In [ ]:
import json, random

with open(COMPILED_PATH, 'r', encoding='utf-8') as f:
    rows = [json.loads(l) for l in f]

random.Random(0).shuffle(rows)
for r in rows[:15]:
    print('Modification:', r['modification'])
    print('Target      :', r['target'])
    for a in r['atoms']:
        print('  ', a['operation'], '| source:', a['source_state'], '| target:', a['target_state'])
    print('parse_ok    :', r['parse_ok'])
    print('---')

## Bước 5 — Chạy E0 → Bảng 4 (mAP@5/@10/@25/@50)

**Chỉ chạy sau khi audit Bước 4 đạt ≥85% Correct.** Chấm điểm theo lô trên GPU nên chỉ mất vài phút.

In [ ]:
# Giai phong VRAM cua Qwen truoc khi chay tinh diem
del compiler_model, compiler_processor
import gc; gc.collect(); torch.cuda.empty_cache()

!python -m src.p0.run_e0 \
    --dataset circo --split val --data-root {DATA_DIR} \
    --feature-cache-dir {FEATURE_CACHE_DIR} \
    --compiled-queries-path {COMPILED_PATH} 2>&1 | tee /workspace/outputs/e0_result.txt

## Bước 6 — Lấy kết quả về rồi Destroy máy

Tải các file trong `/workspace/outputs/` (`circo_compiled.jsonl`, `e0_result.txt`) về máy qua trình duyệt file của Jupyter (chuột phải → Download). Cache đặc trưng 11.7GB không cần tải, tính lại được. **Xong thì Destroy máy trên trang Vast để dừng tính tiền.**

In [ ]:
!ls -la /workspace/outputs/